# Exploratory Data Analysis — Superimposed Preeclampsia Prediction

This notebook explores the synthetic EHR cohort used to predict **superimposed preeclampsia (PE)**  
in pregnant women with a prior diagnosis of chronic hypertension.

## Cohort Overview

| Attribute | Details |
|-----------|--------|
| **Index condition** | Chronic hypertension (ICD-10: O10, I10) |
| **Outcome** | Superimposed preeclampsia (ICD-10: O11) |
| **Age range** | 12–50 years at delivery |
| **Data type** | Synthetic EHR — demographics, vitals, labs, obstetric history |

> **Note:** This notebook uses a synthetic dataset generated to mirror the structure of a real  
> chronic-hypertension pregnancy cohort. No real patient data is used.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data_preprocessing import EHRPreprocessor
from src.feature_engineering import FeatureEngineer
from src.visualization import ResultsVisualizer

sns.set_theme(style='whitegrid', font_scale=1.1)
PALETTE = ['#4878CF', '#D65F5F']
TARGET  = 'superimposed_pe'
SEED    = 42

## 1. Load Dataset

In [ ]:
import subprocess, sys
from pathlib import Path

data_path = Path('../data/synthetic/synthetic_pe_cohort.csv')
if not data_path.exists():
    print('Generating synthetic dataset ...')
    subprocess.run([sys.executable, '../data/synthetic/generate_synthetic_data.py'], check=True)

df = pd.read_csv(data_path)
print(f'Shape: {df.shape}')
df.head()

## 2. Dataset Overview

In [ ]:
prep = EHRPreprocessor(target_col=TARGET)
summary = prep.summary(df)
print(summary.to_string())

In [ ]:
print('Missing value summary:')
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False).plot(
    kind='barh', figsize=(7, 4), color='#4878CF'
)
plt.xlabel('Count')
plt.title('Missing Values per Column')
plt.tight_layout()
plt.show()

## 3. Outcome Distribution

In [ ]:
counts = df[TARGET].value_counts()
prevalence = df[TARGET].mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
axes[0].bar(['No PE', 'Superimposed PE'], counts.values,
            color=PALETTE, edgecolor='white', width=0.5)
for i, (v, c) in enumerate(zip(['No PE', 'Superimposed PE'], counts.values)):
    axes[0].text(i, c + 10, f'n={c}\n({c/len(df):.1%})',
                 ha='center', fontsize=11)
axes[0].set_title(f'Superimposed PE Prevalence = {prevalence:.1%}', fontweight='bold')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=['No PE', 'Superimposed PE'],
            autopct='%1.1f%%', colors=PALETTE, startangle=90)
axes[1].set_title('Outcome Distribution', fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Superimposed PE cases: {counts[1]}  |  Controls: {counts[0]}  |  Prevalence: {prevalence:.1%}')

## 4. Demographic Characteristics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Age distribution
for val, label, color in [(0, 'No PE', PALETTE[0]), (1, 'Superimposed PE', PALETTE[1])]:
    axes[0,0].hist(df.loc[df[TARGET]==val, 'age_at_delivery'].dropna(),
                   bins=20, alpha=0.6, color=color, label=label)
axes[0,0].set_title('Age at Delivery Distribution')
axes[0,0].set_xlabel('Age (years)')
axes[0,0].legend()

# Race/ethnicity
race_pe = df.groupby(['race_ethnicity', TARGET]).size().unstack(fill_value=0)
race_pe.plot(kind='bar', ax=axes[0,1], color=PALETTE, edgecolor='white')
axes[0,1].set_title('Race/Ethnicity by PE Status')
axes[0,1].set_xlabel('')
axes[0,1].tick_params(axis='x', rotation=30)
axes[0,1].legend(['No PE', 'Superimposed PE'])

# Insurance type
ins_pe = df.groupby(['insurance_type', TARGET]).size().unstack(fill_value=0)
ins_pe.plot(kind='bar', ax=axes[1,0], color=PALETTE, edgecolor='white')
axes[1,0].set_title('Insurance Type by PE Status')
axes[1,0].set_xlabel('')
axes[1,0].tick_params(axis='x', rotation=15)
axes[1,0].legend(['No PE', 'Superimposed PE'])

# BMI
for val, label, color in [(0, 'No PE', PALETTE[0]), (1, 'Superimposed PE', PALETTE[1])]:
    axes[1,1].hist(df.loc[df[TARGET]==val, 'bmi_prepregnancy'].dropna(),
                   bins=25, alpha=0.6, color=color, label=label)
axes[1,1].axvline(30, color='red', linestyle='--', alpha=0.5, label='BMI=30')
axes[1,1].set_title('Pre-pregnancy BMI')
axes[1,1].set_xlabel('BMI (kg/m²)')
axes[1,1].legend()

plt.suptitle('Demographic Characteristics by Outcome', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Clinical History — Risk Factors

In [ ]:
binary_features = [
    'prior_pe', 'pregestational_diabetes', 'gestational_diabetes',
    'renal_disease', 'autoimmune_disorder', 'prior_preterm_birth',
    'smoking_status', 'multiple_gestation',
]

# Prevalence per group
risk_table = pd.DataFrame()
for feat in binary_features:
    for group, label in [(0, 'No PE'), (1, 'Superimposed PE')]:
        subset = df[df[TARGET] == group][feat].dropna()
        risk_table.loc[feat, label] = subset.mean() * 100

risk_table = risk_table.sort_values('Superimposed PE', ascending=False)

risk_table.plot(kind='barh', figsize=(9, 5), color=PALETTE[::-1], edgecolor='white')
plt.xlabel('Prevalence (%)')
plt.title('Prevalence of Risk Factors by PE Status', fontweight='bold')
plt.legend(['Superimposed PE', 'No PE'])
plt.tight_layout()
plt.show()

print(risk_table.round(1).to_string())

In [ ]:
# Chi-square tests for binary risk factors
from scipy.stats import chi2_contingency, kruskal

print('Chi-square tests (binary features):')
print(f'  {"Feature":<35} {"PE%":>6}  {"noPE%":>6}  {"χ²":>7}  {"p-value":>10}')
print('  ' + '-'*70)
for feat in binary_features:
    ct = pd.crosstab(df[feat].fillna(0).astype(int), df[TARGET])
    chi2, p, dof, _ = chi2_contingency(ct)
    pe_pct  = df[df[TARGET]==1][feat].mean() * 100
    nope_pct= df[df[TARGET]==0][feat].mean() * 100
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f'  {feat:<35} {pe_pct:>5.1f}%  {nope_pct:>5.1f}%  {chi2:>7.2f}  {p:>10.4f} {sig}')

## 6. Blood Pressure Analysis

In [ ]:
bp_features = ['sbp_first_trimester', 'dbp_first_trimester',
               'sbp_second_trimester', 'dbp_second_trimester',
               'sbp_max_antepartum', 'dbp_max_antepartum']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, feat in zip(axes, bp_features):
    data_no  = df.loc[df[TARGET]==0, feat].dropna()
    data_yes = df.loc[df[TARGET]==1, feat].dropna()
    stat, p  = kruskal(data_no, data_yes)
    
    ax.boxplot([data_no, data_yes], labels=['No PE', 'PE'], patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='black', linewidth=2))
    ax.set_title(f'{feat}\nKruskal-Wallis p={p:.4f}', fontsize=9)
    ax.set_ylabel('mmHg')

plt.suptitle('Blood Pressure by Outcome', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Laboratory Values

In [ ]:
lab_features = [
    'protein_urine_24h_mg', 'creatinine_mg_dl', 'platelet_count_k_ul',
    'alt_u_l', 'ast_u_l', 'uric_acid_mg_dl', 'hemoglobin_g_dl', 'ldh_u_l'
]

lab_stats = []
for feat in lab_features:
    no_pe  = df.loc[df[TARGET]==0, feat].dropna()
    yes_pe = df.loc[df[TARGET]==1, feat].dropna()
    stat, p = kruskal(no_pe, yes_pe)
    lab_stats.append({
        'Lab':        feat,
        'No PE (median [IQR])': f"{no_pe.median():.1f} [{no_pe.quantile(0.25):.1f}–{no_pe.quantile(0.75):.1f}]",
        'PE (median [IQR])':    f"{yes_pe.median():.1f} [{yes_pe.quantile(0.25):.1f}–{yes_pe.quantile(0.75):.1f}]",
        'p-value': round(p, 4),
        'Sig': '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '',
    })

lab_df = pd.DataFrame(lab_stats)
print('Laboratory Values by PE Status (Kruskal-Wallis test):')
print(lab_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, feat in zip(axes, lab_features):
    data_no  = df.loc[df[TARGET]==0, feat].dropna()
    data_yes = df.loc[df[TARGET]==1, feat].dropna()
    _, p = kruskal(data_no, data_yes)
    
    ax.violinplot([data_no, data_yes], positions=[1, 2], showmedians=True)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['No PE', 'PE'], fontsize=9)
    ax.set_title(f'{feat}\np={p:.4f}', fontsize=8.5)

plt.suptitle('Laboratory Values by Outcome (Violin Plot)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Feature Correlations

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
corr = df[numeric_cols].corr(method='spearman')

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.3, ax=ax,
            cbar_kws={'label': 'Spearman ρ'})
ax.set_title('Feature Correlation Matrix (Spearman)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlates with outcome
target_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print('\nTop 15 features correlated with superimposed PE:')
print(target_corr.head(15).to_string())

## 9. Feature Engineering & Ranking

In [ ]:
fe = FeatureEngineer(random_state=SEED)
df_eng = fe.add_clinical_features(df)
print(f'Features after engineering: {len(df_eng.columns)} (added {len(df_eng.columns)-len(df.columns)})')

# Quick preprocessing for feature ranking
prep2 = EHRPreprocessor(
    target_col='superimposed_pe',
    imputation_strategy='median',
    scaling='none',            # keep raw scale for ranking
    encoding_method='ordinal',
)
X_rank, y_rank, feat_names = prep2.fit_transform(df_eng)

ranking = fe.rank_features(X_rank, y_rank, feat_names)
print('\nTop 20 features by mutual information:')
print(ranking.head(20)[['feature', 'mutual_info', 'anova_f', 'spearman_r']].to_string(index=False))

In [ ]:
top20 = ranking.head(20)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top20_sorted_mi = top20.sort_values('mutual_info')
axes[0].barh(top20_sorted_mi['feature'], top20_sorted_mi['mutual_info'], color='#4878CF', alpha=0.85)
axes[0].set_title('Top 20 Features — Mutual Information', fontweight='bold')
axes[0].set_xlabel('Mutual Information')

top20_sorted_sp = top20.sort_values('spearman_abs_r')
colors = ['#D65F5F' if r > 0 else '#4878CF' for r in top20_sorted_sp['spearman_r']]
axes[1].barh(top20_sorted_sp['feature'], top20_sorted_sp['spearman_r'], color=colors, alpha=0.85)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Top 20 Features — Spearman Correlation with PE', fontweight='bold')
axes[1].set_xlabel('Spearman ρ')

plt.tight_layout()
plt.show()

## 10. Cohort Summary Table (Table 1)

In [ ]:
# Generate a clinical Table 1 comparing PE vs no-PE
continuous_feats = [
    'age_at_delivery', 'bmi_prepregnancy', 'gestational_age_at_delivery',
    'parity', 'gravidity', 'chronic_hypertension_duration_yrs',
    'sbp_max_antepartum', 'dbp_max_antepartum',
    'protein_urine_24h_mg', 'platelet_count_k_ul', 'creatinine_mg_dl',
    'ast_u_l', 'uric_acid_mg_dl',
]

no_pe  = df[df[TARGET] == 0]
yes_pe = df[df[TARGET] == 1]

table1_rows = []
for feat in continuous_feats:
    no_vals  = no_pe[feat].dropna()
    yes_vals = yes_pe[feat].dropna()
    stat, p  = kruskal(no_vals, yes_vals)
    table1_rows.append({
        'Variable': feat,
        f'No PE (n={len(no_pe)})':   f"{no_vals.median():.1f} [{no_vals.quantile(0.25):.1f}–{no_vals.quantile(0.75):.1f}]",
        f'PE (n={len(yes_pe)})':     f"{yes_vals.median():.1f} [{yes_vals.quantile(0.25):.1f}–{yes_vals.quantile(0.75):.1f}]",
        'p-value': round(p, 3),
    })

table1 = pd.DataFrame(table1_rows)
print('Table 1. Cohort Characteristics (median [IQR])')
print(table1.to_string(index=False))

## 11. Key EDA Findings

| Finding | Detail |
|---------|--------|
| **Prevalence** | Superimposed PE ~25% in chronic hypertension cohort |
| **Race disparity** | Non-Hispanic Black patients show higher PE prevalence |
| **Prior PE** | Strongest binary risk factor (χ² significant) |
| **Blood pressure** | Max antepartum SBP significantly elevated in PE group |
| **Proteinuria** | 24-h urine protein strongly differentiates groups |
| **Platelets** | Thrombocytopenia trend in PE group |
| **Liver enzymes** | AST/ALT elevation in PE group (HELLP risk) |
| **Missing data** | ~5–12% for some lab values; handled by imputation |

---
*Proceed to `02_model_training_comparison.ipynb` for model training and evaluation.*